# 03 Baseline Model: IEEE-CIS Fraud Detection

Это `Jupyter notebook` для первого baseline моделирования после `EDA`, `SQL` и `feature ideas`.

Цель:
- собрать простой baseline pipeline;
- обучить первую понятную модель;
- посчитать базовые anti-fraud метрики;
- научиться интерпретировать результат в бизнес-контексте.


## План работы

1. Загрузить `train_transaction` и `train_identity`.
2. Собрать тот же MVP feature set, что и в `02_feature_ideas.ipynb`.
3. Выделить `X` и `y`.
4. Сделать `train / validation split`.
5. Обучить `LogisticRegression` как baseline.
6. Посчитать `precision`, `recall`, `f1`, `roc_auc` и confusion matrix.
7. Коротко интерпретировать, что получилось и что улучшать дальше.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [2]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
TRANSACTION_PATH = DATA_DIR / 'train_transaction.csv'
IDENTITY_PATH = DATA_DIR / 'train_identity.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRANSACTION_PATH exists:', TRANSACTION_PATH.exists())
print('IDENTITY_PATH exists:', IDENTITY_PATH.exists())


PROJECT_ROOT: /Users/drhtka/Downloads/Projects/Llm_ml_RAG/anti_fraud_analytics_platform
TRANSACTION_PATH exists: True
IDENTITY_PATH exists: True


In [3]:
def load_csv_if_exists(path: Path):
    if path.exists():
        print(f'Loaded: {path.name}')
        return pd.read_csv(path)
    print(f'File not found: {path}')
    return None


transactions = load_csv_if_exists(TRANSACTION_PATH)
identity = load_csv_if_exists(IDENTITY_PATH)


Loaded: train_transaction.csv
Loaded: train_identity.csv


## Сборка baseline feature set

Берем только простые и уже понятные признаки из `02_feature_ideas.ipynb`.

Это важно, потому что сейчас цель не максимальное качество, а понятный baseline.

На этом шаге добавляем только `1` новый behavioral feature, чтобы улучшать baseline маленькими шагами.

Что именно изучаем:
- как превратить историю активности сущности в числовой признак;
- как добавить такой признак без leakage;
- помогает ли даже простой velocity proxy улучшить anti-fraud baseline.

Для MVP берем максимально простой behavioral signal: сколько транзакций связано с `card1` в train-части. Это не идеальная business entity, но для текущего этапа это понятный proxy активности, который легко объяснить на собеседовании.


In [4]:
high_risk_p_domains = {'outlook.com'}
high_risk_r_domains = {'outlook.com', 'icloud.com', 'gmail.com'}

if transactions is not None:
    feature_df = transactions[[
        'TransactionID', 'isFraud', 'TransactionAmt', 'ProductCD', 'card1', 'card4', 'card6',
        'P_emaildomain', 'R_emaildomain', 'addr1', 'TransactionDT'
    ]].copy()

    feature_df['feat_productcd_c_flag'] = (feature_df['ProductCD'] == 'C').astype(int)
    feature_df['feat_card4_discover_flag'] = (feature_df['card4'] == 'discover').astype(int)
    feature_df['feat_card6_credit_flag'] = (feature_df['card6'] == 'credit').astype(int)
    feature_df['feat_high_risk_p_email_flag'] = feature_df['P_emaildomain'].isin(high_risk_p_domains).astype(int)
    feature_df['feat_high_risk_r_email_flag'] = feature_df['R_emaildomain'].isin(high_risk_r_domains).astype(int)
    feature_df['feat_missing_r_email_flag'] = feature_df['R_emaildomain'].isna().astype(int)
    feature_df['feat_amount_log1p'] = np.log1p(feature_df['TransactionAmt'])

    base_feature_cols = [
        'feat_productcd_c_flag',
        'feat_high_risk_r_email_flag',
        'feat_card6_credit_flag',
        'feat_high_risk_p_email_flag',
        'feat_card4_discover_flag',
        'feat_missing_r_email_flag',
        'feat_amount_log1p',
    ]

    display(feature_df[base_feature_cols + ['isFraud']].head())


,feat_productcd_c_flag,feat_high_risk_r_email_flag,feat_card6_credit_flag,feat_high_risk_p_email_flag,feat_card4_discover_flag,feat_missing_r_email_flag,feat_amount_log1p,isFraud
0,0,0,1,0,1,1,4.241327,0
1,0,0,1,0,0,1,3.401197,0
2,0,0,0,1,0,1,4.094345,0
3,0,0,0,0,0,1,3.931826,0
4,0,0,1,0,0,1,3.931826,0


## Подготовка `X` и `y`

Здесь мы явно отделяем признаки от target.


In [5]:
y = feature_df['isFraud'].copy()

print('feature_df shape:', feature_df.shape)
print('y shape:', y.shape)
print('fraud rate:', round(100 * y.mean(), 3), '%')


feature_df shape: (590540, 18)
y shape: (590540,)
fraud rate: 3.499 %


## Train / validation split

Используем `stratify=y`, потому что fraud class сильно несбалансирован.


In [6]:
train_df, valid_df = train_test_split(
    feature_df,
    test_size=0.2,
    random_state=42,
    stratify=feature_df['isFraud'],
)

train_df = train_df.copy()
valid_df = valid_df.copy()

y_train = train_df['isFraud'].copy()
y_valid = valid_df['isFraud'].copy()

print('train_df:', train_df.shape)
print('valid_df:', valid_df.shape)
print('train fraud rate:', round(100 * y_train.mean(), 3), '%')
print('valid fraud rate:', round(100 * y_valid.mean(), 3), '%')


train_df: (472432, 18)
valid_df: (118108, 18)
train fraud rate: 3.499 %
valid fraud rate: 3.499 %


## Baseline model

Берем `LogisticRegression` как самый понятный baseline.

Перед обучением добавляем один простой behavioral feature: `feat_card1_txn_count_log1p`.

Что это значит простыми словами:
- мы считаем, насколько активен `card1` в train-части;
- затем передаем модели логарифм этого количества транзакций;
- это первый простой proxy для transaction velocity / activity intensity.

Почему делаем именно так:
- это маленький MVP-шаг без сложных временных окон;
- признак легко объяснить;
- train-only расчет сохраняет честность baseline.

Используем `class_weight='balanced'`, чтобы модель не игнорировала редкий fraud class.


In [7]:
card1_amount_stats = (
    train_df.groupby('card1')['TransactionAmt']
    .agg(card1_amt_mean='mean', card1_amt_std='std')
)

card1_txn_count = train_df.groupby('card1').size().rename('card1_txn_count')

train_df = train_df.join(card1_amount_stats, on='card1').join(card1_txn_count, on='card1').copy()
valid_df = valid_df.join(card1_amount_stats, on='card1').join(card1_txn_count, on='card1').copy()

for current_df in (train_df, valid_df):
    current_threshold = current_df['card1_amt_mean'] + 3 * current_df['card1_amt_std'].fillna(0)
    current_df['feat_amount_gt_card1_avg_plus_3std'] = (
        current_df['TransactionAmt'] > current_threshold
    ).astype(int)
    current_df['feat_card1_txn_count_log1p'] = np.log1p(current_df['card1_txn_count'].fillna(0))

initial_feature_cols = base_feature_cols + [
    'feat_amount_gt_card1_avg_plus_3std',
    'feat_card1_txn_count_log1p',
]

X_train = train_df[initial_feature_cols].copy()
X_valid = valid_df[initial_feature_cols].copy()

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)

baseline_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',
)

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_valid)
y_proba = baseline_model.predict_proba(X_valid)[:, 1]


X_train: (472432, 9)
X_valid: (118108, 9)


## Метрики

Для anti-fraud baseline нам важны не только accuracy, а прежде всего:
- `precision`
- `recall`
- `f1`
- `roc_auc`
- confusion matrix

Как это понимать:

- `precision` показывает, насколько чистый поток fraud alert мы получаем;
- `recall` показывает, какую долю настоящего fraud мы смогли поймать;
- `f1` нужен как простой баланс между `precision` и `recall`;
- `roc_auc` показывает, есть ли вообще полезный сигнал в признаках до выбора конкретного порога;
- confusion matrix переводит проценты в реальные числа ошибок.

Для anti-fraud это важно, потому что:

- низкий `precision` = слишком много ложных тревог и лишней ручной проверки;
- низкий `recall` = много пропущенного fraud;
- хороший baseline это не просто модель с цифрами, а модель, у которой понятен trade-off между fraud capture и false positives.


In [8]:
metrics_summary = pd.DataFrame([
    {
        'precision': precision_score(y_valid, y_pred, zero_division=0),
        'recall': recall_score(y_valid, y_pred, zero_division=0),
        'f1': f1_score(y_valid, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, y_proba),
    }
])

display(metrics_summary)


,precision,recall,f1,roc_auc
0,0.085236,0.61505,0.149723,0.746597


In [9]:
conf_matrix = pd.DataFrame(
    confusion_matrix(y_valid, y_pred),
    index=['actual_non_fraud', 'actual_fraud'],
    columns=['pred_non_fraud', 'pred_fraud'],
)

display(conf_matrix)


,pred_non_fraud,pred_fraud
actual_non_fraud,86694,27281
actual_fraud,1591,2542


In [10]:
print(classification_report(y_valid, y_pred, digits=4, zero_division=0))


              precision    recall  f1-score   support

           0     0.9820    0.7606    0.8573    113975
           1     0.0852    0.6150    0.1497      4133

    accuracy                         0.7555    118108
   macro avg     0.5336    0.6878    0.5035    118108
weighted avg     0.9506    0.7555    0.8325    118108



## Threshold analysis

Сейчас `predict()` использует стандартный threshold `0.5`.

Но в anti-fraud почти всегда полезно посмотреть, как поведение модели меняется при разных порогах:

- lower threshold -> обычно выше `recall`, но больше `false positives`;
- higher threshold -> обычно выше `precision`, но ниже `recall`.

Ниже сравниваем три простых варианта:

- `0.3` как более агрессивный fraud capture;
- `0.5` как baseline threshold;
- `0.7` как более строгий fraud trigger.


In [11]:
threshold_rows = []

for threshold in [0.3, 0.5, 0.7]:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_pred_threshold).ravel()

    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_valid, y_pred_threshold, zero_division=0),
        'recall': recall_score(y_valid, y_pred_threshold, zero_division=0),
        'f1': f1_score(y_valid, y_pred_threshold, zero_division=0),
        'predicted_fraud_count': int(y_pred_threshold.sum()),
        'manual_review_rate_pct': round(100 * y_pred_threshold.mean(), 2),
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'tn': int(tn),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)


,threshold,precision,recall,f1,predicted_fraud_count,manual_review_rate_pct,tp,fp,fn,tn
0,0.3,0.046065,0.886281,0.087578,79518,67.33,3663,75855,470,38120
1,0.5,0.085236,0.615050,0.149723,29823,25.25,2542,27281,1591,86694
2,0.7,0.139336,0.391241,0.205490,11605,9.83,1617,9988,2516,103987


### Как читать threshold table

Когда посмотришь на эту таблицу, обрати внимание на четыре вещи:

1. как меняется `precision` при росте threshold;
2. как меняется `recall` при росте threshold;
3. как сильно меняется `manual_review_rate_pct`;
4. какой trade-off выглядит разумнее для бизнеса.

Если порог ниже:

- модель будет чаще говорить `fraud`;
- fraud capture обычно растет;
- но ложных срабатываний тоже становится больше.

Если порог выше:

- модель становится строже;
- fraud alerts меньше;
- но растет риск пропустить часть fraud.

Именно поэтому в anti-fraud threshold выбирают не "на глаз", а под бизнес-цель: что для нас сейчас важнее, поймать больше fraud или уменьшить ложные тревоги.


## Feature importance proxy

Для `LogisticRegression` можно посмотреть на коэффициенты как на грубую интерпретацию вклада признаков.

Это не финальная explainability, но хороший первый шаг.


In [12]:
coef_df = pd.DataFrame({
    'feature': initial_feature_cols,
    'coefficient': baseline_model.coef_[0],
    'abs_coefficient': np.abs(baseline_model.coef_[0]),
}).sort_values('abs_coefficient', ascending=False)

display(coef_df)


,feature,coefficient,abs_coefficient
0,feat_productcd_c_flag,1.442516,1.442516
1,feat_high_risk_r_email_flag,0.986521,0.986521
2,feat_card6_credit_flag,0.627228,0.627228
4,feat_card4_discover_flag,0.563667,0.563667
7,feat_amount_gt_card1_avg_plus_3std,-0.357325,0.357325
6,feat_amount_log1p,0.348335,0.348335
3,feat_high_risk_p_email_flag,0.198336,0.198336
5,feat_missing_r_email_flag,-0.142842,0.142842
8,feat_card1_txn_count_log1p,0.049649,0.049649


## Короткая интерпретация

После запуска этого ноутбука ответь себе на вопросы:

1. Какие признаки реально помогают baseline модели?
2. Что важнее в текущем baseline: `precision` или `recall`?
3. Какие ошибки модель делает чаще всего?
4. Что стоит улучшить следующим шагом: признаки, порог или сам алгоритм?


## Что делать дальше

Если baseline отработал:

- кратко зафиксируй метрики;
- посмотри, помог ли один простой behavioral feature;
- опиши 2-3 сильных признака;
- опиши 2 главных слабости baseline;
- потом можно сравнить `LogisticRegression` с еще одной простой моделью.

Но пока не распыляемся: сначала нужно понять и объяснить именно этот baseline.


## First baseline interpretation

`LogisticRegression` с простым MVP feature set и одним behavioral feature показывает, что в данных уже есть полезный fraud signal: `roc_auc` остается около `0.745`, а `recall` остается около `0.61`, то есть модель все еще находит примерно 61% fraud cases.

При этом `precision` остается низким, то есть модель все еще генерирует слишком много false positives. В текущем виде такой baseline скорее подходит как стартовая модель для manual review, а не для автоматической блокировки.

## Threshold analysis

По умолчанию `predict()` использует threshold `0.5`.

Но в anti-fraud важно посмотреть, как меняется поведение модели при разных порогах:

- ниже threshold -> обычно выше `recall`, но больше `false positives`
- выше threshold -> обычно выше `precision`, но ниже `recall`

Сейчас сравним три простых варианта:

- `0.3`
- `0.5`
- `0.7`

Смотри потом на 4 вещи:

1. как меняется `precision`
2. как меняется `recall`
3. как меняется `manual_review_rate_pct`
4. какой порог выглядит реалистичнее для бизнеса

In [13]:
threshold_rows = []

for threshold in [0.3, 0.5, 0.7]:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_pred_threshold).ravel()

    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_valid, y_pred_threshold, zero_division=0),
        'recall': recall_score(y_valid, y_pred_threshold, zero_division=0),
        'f1': f1_score(y_valid, y_pred_threshold, zero_division=0),
        'predicted_fraud_count': int(y_pred_threshold.sum()),
        'manual_review_rate_pct': round(100 * y_pred_threshold.mean(), 2),
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'tn': int(tn),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

,threshold,precision,recall,f1,predicted_fraud_count,manual_review_rate_pct,tp,fp,fn,tn
0,0.3,0.046065,0.886281,0.087578,79518,67.33,3663,75855,470,38120
1,0.5,0.085236,0.615050,0.149723,29823,25.25,2542,27281,1591,86694
2,0.7,0.139336,0.391241,0.205490,11605,9.83,1617,9988,2516,103987


## Threshold takeaway

После сравнения `0.3`, `0.5` и `0.7` нужно ответить:

- какой threshold лучше для `manual review`
- какой threshold слишком шумный
- какой threshold начинает слишком сильно ронять `recall`


Сравнение порогов показало, что:

- `0.3` дает очень высокий `recall`, но слишком высокий `manual_review_rate`, поэтому для MVP выглядит слишком шумным;
- `0.5` остается полезной baseline-точкой сравнения, но все еще создает много false positives;
- `0.7` выглядит более реалистичным рабочим порогом для `manual review`, потому что заметно снижает review load и повышает `precision`, хотя и ухудшает `recall`.

Для текущего MVP baseline разумно рассматривать `0.7` как рабочий threshold для fraud review, а `0.5` использовать как reference point для сравнения.

## Baseline summary

### Что получилось

Обновленный `LogisticRegression` baseline с одним простым behavioral feature показал, что даже такой маленький шаг не ломает честную baseline-оценку и сохраняет полезный fraud signal:

- `roc_auc` остается около `0.745`
- при threshold `0.5` модель по-прежнему находит около `61%` fraud cases
- `precision` улучшается совсем немного, поэтому false positives все еще остаются главной проблемой

### Что показал threshold analysis

Сравнение порогов `0.3`, `0.5` и `0.7` показало:

- threshold `0.3` слишком агрессивный: fraud capture высокий, но `manual_review_rate` слишком большой
- threshold `0.5` остается хорошей baseline-точкой сравнения
- threshold `0.7` выглядит более реалистичным рабочим вариантом для `manual review`, потому что снижает review load и повышает `precision`, хотя и теряет часть fraud

### Сильные стороны baseline

- модель уже умеет отделять fraud от non-fraud лучше случайного уровня
- самые понятные признаки действительно дают полезный сигнал
- baseline уже можно объяснять в терминах бизнеса и anti-fraud логики

### Слабые стороны baseline

- слишком много false positives
- даже после добавления одного behavioral feature set все еще очень простой
- часть признаков построена на proxy-полях, а не на идеальных business entities

### Текущий вывод

В текущем виде baseline скорее подходит как стартовая модель для `manual review`, а не для автоматической блокировки.

### Что улучшать дальше

Следующие логичные улучшения:

1. добавить еще 1-2 более сильных behavioral features
2. аккуратнее проработать threshold под бизнес-цель
3. сравнить baseline с еще одной простой моделью
4. перейти от rule flags к более богатому feature engineering